In [2]:
# =============================================================================
#  CUSTOMER CHURN PREDICTOR
#  Dataset : Telco Customer Churn (IBM / Kaggle)
#  Models  : Random Forest · XGBoost · LightGBM
#  Goal    : Predict which customers will churn + identify top churn drivers
# =============================================================================

import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import os, time, urllib.request

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing   import StandardScaler, LabelEncoder
from sklearn.pipeline        import Pipeline
from sklearn.compose         import ColumnTransformer
from sklearn.preprocessing   import OneHotEncoder
from sklearn.ensemble        import RandomForestClassifier
from sklearn.metrics         import (f1_score, accuracy_score, roc_auc_score,
                                     confusion_matrix, classification_report,
                                     roc_curve, precision_recall_curve,
                                     average_precision_score)
from xgboost     import XGBClassifier
import lightgbm  as lgb
import mlflow.xgboost
import mlflow.lightgbm
import mlflow, mlflow.sklearn


# ── Style ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#FAFAF9',
    'axes.spines.top': False,    'axes.spines.right': False,
    'axes.grid': True,           'grid.color': '#EBEBEA',
    'grid.linewidth': 0.7,       'font.family': 'sans-serif', 'font.size': 11,
})
TEAL  = '#0F6E56'; CORAL = '#993C1D'; BLUE  = '#185FA5'
AMBER = '#854F0B'; GRAY  = '#888780'; LIGHT = '#E1F5EE'
os.makedirs('plots', exist_ok=True)

print("=" * 65)
print("  TELCO CUSTOMER CHURN PREDICTOR")
print("  EDA → Feature Engineering → 3 Models → Business Insights")
print("=" * 65)


# =============================================================================
# SECTION 1 — DATA LOADING
# =============================================================================
print("\n[1/8] Loading data...")

def load_data():
    if os.path.exists('telco_churn.csv'):
        print("  Source: telco_churn.csv (local)")
        return pd.read_csv('telco_churn.csv')
    url = ('https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d'
           '/master/data/Telco-Customer-Churn.csv')
    print("  Source: downloading from GitHub...")
    urllib.request.urlretrieve(url, 'telco_churn.csv')
    return pd.read_csv('telco_churn.csv')

df_raw = load_data()
print(f"  Shape        : {df_raw.shape[0]:,} customers × {df_raw.shape[1]} features")
print(f"  Churn rate   : {df_raw['Churn'].value_counts(normalize=True)['Yes']*100:.1f}%")
print(f"  Missing vals : {df_raw.isnull().sum().sum()}")


# =============================================================================
# SECTION 2 — DATA CLEANING
# =============================================================================
print("\n[2/8] Cleaning data...")

df = df_raw.copy()
df.drop('customerID', axis=1, inplace=True)

# TotalCharges has spaces instead of NaN for new customers (tenure=0)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
n_missing = df['TotalCharges'].isna().sum()
print(f"  TotalCharges NaN (new customers): {n_missing} → filling with MonthlyCharges")
df['TotalCharges'].fillna(df['MonthlyCharges'], inplace=True)

# Encode binary target
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

print(f"  Churn = 1: {df['Churn'].sum():,}  ({df['Churn'].mean()*100:.1f}%)")
print(f"  Churn = 0: {(1-df['Churn']).sum():,}  ({(1-df['Churn']).mean()*100:.1f}%)")
print(f"  ⚠  Class imbalance present — will use F1 score and class_weight")


# =============================================================================
# SECTION 3 — EXPLORATORY DATA ANALYSIS
# =============================================================================
print("\n[3/8] Running EDA → plots/01_eda_overview.png ...")

NUMERICAL = ['tenure', 'MonthlyCharges', 'TotalCharges']
BINARY    = ['gender','Partner','Dependents','PhoneService',
             'PaperlessBilling','SeniorCitizen']
MULTI_CAT = ['MultipleLines','InternetService','OnlineSecurity','OnlineBackup',
             'DeviceProtection','TechSupport','StreamingTV','StreamingMovies',
             'Contract','PaymentMethod']

fig = plt.figure(figsize=(20, 16))
gs  = gridspec.GridSpec(4, 4, figure=fig, hspace=0.55, wspace=0.38)

# ── Churn distribution ────────────────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 0])
counts = df['Churn'].value_counts()
bars   = ax.bar(['No Churn', 'Churned'], counts.values,
                color=[TEAL, CORAL], edgecolor='white', linewidth=0.5)
ax.set_title('Churn distribution', fontweight='500')
ax.set_ylabel('Customers')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
            f'{val:,}\n({val/len(df)*100:.1f}%)',
            ha='center', fontsize=10, fontweight='500')

# ── Tenure distribution by churn ──────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 1])
ax.hist(df[df['Churn']==0]['tenure'], bins=30, alpha=0.7, color=TEAL,   label='No churn')
ax.hist(df[df['Churn']==1]['tenure'], bins=30, alpha=0.7, color=CORAL,  label='Churned')
ax.set_title('Tenure by churn status', fontweight='500')
ax.set_xlabel('Months as customer'); ax.set_ylabel('Count'); ax.legend(fontsize=9)

# ── Monthly charges by churn ───────────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 2])
df.boxplot(column='MonthlyCharges', by='Churn', ax=ax,
           boxprops=dict(color=BLUE), medianprops=dict(color=CORAL, linewidth=2))
ax.set_title('Monthly charges by churn', fontweight='500')
ax.set_xlabel('Churn (0=No, 1=Yes)'); ax.set_ylabel('Monthly charges ($)')
plt.sca(ax); plt.title('Monthly charges by churn', fontweight='500')

# ── Contract type vs churn rate ───────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 3])
contract_churn = df.groupby('Contract')['Churn'].mean().sort_values(ascending=False)
colors_bar = [CORAL if v > 0.2 else TEAL for v in contract_churn.values]
ax.barh(contract_churn.index, contract_churn.values * 100, color=colors_bar)
ax.set_xlabel('Churn rate (%)'); ax.set_title('Churn rate by contract type', fontweight='500')
for i, v in enumerate(contract_churn.values):
    ax.text(v*100 + 0.3, i, f'{v*100:.1f}%', va='center', fontsize=10)

# ── Internet service vs churn rate ────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 0])
isp_churn = df.groupby('InternetService')['Churn'].mean().sort_values(ascending=False)
ax.barh(isp_churn.index, isp_churn.values * 100,
        color=[CORAL, AMBER, TEAL])
ax.set_xlabel('Churn rate (%)'); ax.set_title('Churn rate by internet service', fontweight='500')
for i, v in enumerate(isp_churn.values):
    ax.text(v*100 + 0.3, i, f'{v*100:.1f}%', va='center', fontsize=10)

# ── Payment method vs churn rate ──────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 1])
pm_churn = df.groupby('PaymentMethod')['Churn'].mean().sort_values(ascending=False)
short_labels = [p.replace(' (automatic)', '\n(auto)').replace(' check', '\ncheck')
                for p in pm_churn.index]
ax.barh(short_labels, pm_churn.values * 100,
        color=[CORAL if v > 0.2 else TEAL for v in pm_churn.values])
ax.set_xlabel('Churn rate (%)'); ax.set_title('Churn rate by payment method', fontweight='500')
ax.tick_params(axis='y', labelsize=9)

# ── Tenure KDE for churned vs retained ────────────────────────────────────────
ax = fig.add_subplot(gs[1, 2])
for label, color, name in [(0, TEAL, 'Retained'), (1, CORAL, 'Churned')]:
    subset = df[df['Churn']==label]['tenure']
    subset.plot.kde(ax=ax, color=color, linewidth=2, label=f'{name} (mean={subset.mean():.0f}mo)')
ax.set_xlabel('Tenure (months)'); ax.set_title('Tenure density by churn', fontweight='500')
ax.legend(fontsize=9)

# ── Senior citizen churn ──────────────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 3])
sc_churn = df.groupby('SeniorCitizen')['Churn'].mean()
ax.bar(['Non-Senior', 'Senior'], sc_churn.values * 100,
       color=[TEAL, CORAL], edgecolor='white')
ax.set_ylabel('Churn rate (%)'); ax.set_title('Churn: senior vs non-senior', fontweight='500')
for i, v in enumerate(sc_churn.values):
    ax.text(i, v*100 + 0.3, f'{v*100:.1f}%', ha='center', fontsize=11, fontweight='500')

# ── Churn rate for value-added services ───────────────────────────────────────
ax = fig.add_subplot(gs[2, :2])
services = ['OnlineSecurity','OnlineBackup','DeviceProtection',
            'TechSupport','StreamingTV','StreamingMovies']
churn_with    = [df[df[s]=='Yes']['Churn'].mean()*100 for s in services]
churn_without = [df[df[s]=='No']['Churn'].mean()*100 for s in services]
x = np.arange(len(services))
width = 0.35
ax.bar(x - width/2, churn_with,    width, label='Has service', color=TEAL,  alpha=0.85)
ax.bar(x + width/2, churn_without, width, label='No service',  color=CORAL, alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels([s.replace('Online','') for s in services],
                                       rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Churn rate (%)'); ax.set_title('Churn rate: with vs without each service',
                                               fontweight='500')
ax.legend(fontsize=9)

# ── MonthlyCharges vs Tenure coloured by Churn ───────────────────────────────
ax = fig.add_subplot(gs[2, 2:])
scatter = ax.scatter(df['tenure'], df['MonthlyCharges'],
                     c=df['Churn'], cmap='RdYlGn_r', s=5, alpha=0.4)
plt.colorbar(scatter, ax=ax, label='Churn', pad=0.01)
ax.set_xlabel('Tenure (months)'); ax.set_ylabel('Monthly charges ($)')
ax.set_title('Churn in tenure × monthly charges space\n(red = churned)',
             fontweight='500')

# ── Correlation of features with churn ───────────────────────────────────────
ax = fig.add_subplot(gs[3, :])
df_enc = df.copy()
for col in df_enc.select_dtypes('object').columns:
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col])
corr_churn = df_enc.corr()['Churn'].drop('Churn').sort_values()
colors_c   = [CORAL if v > 0 else TEAL for v in corr_churn.values]
ax.barh(corr_churn.index, corr_churn.values, color=colors_c)
ax.axvline(0, color=GRAY, linewidth=0.8, linestyle='--')
ax.set_xlabel('Pearson correlation with Churn')
ax.set_title('All features: correlation with churn (red = increases churn risk)',
             fontweight='500')
ax.tick_params(axis='y', labelsize=9)

fig.suptitle('Telco Customer Churn — Exploratory Data Analysis',
             fontsize=15, fontweight='500', y=1.01)
plt.savefig('plots/01_eda_overview.png', dpi=150, bbox_inches='tight',
            facecolor='white')
plt.close()
print("  Saved → plots/01_eda_overview.png")


# =============================================================================
# SECTION 4 — FEATURE ENGINEERING
# =============================================================================
print("\n[4/8] Feature engineering...")

df_feat = df.copy()

# New features based on business logic
df_feat['charges_per_month_of_tenure'] = (
    df_feat['TotalCharges'] / (df_feat['tenure'] + 1)
)
df_feat['is_new_customer']     = (df_feat['tenure'] <= 6).astype(int)
df_feat['is_loyal_customer']   = (df_feat['tenure'] >= 48).astype(int)
df_feat['has_no_services']     = (
    (df_feat['OnlineSecurity'] == 'No') &
    (df_feat['OnlineBackup']   == 'No') &
    (df_feat['TechSupport']    == 'No') &
    (df_feat['DeviceProtection'] == 'No')
).astype(int)
df_feat['n_streaming_services'] = (
    (df_feat['StreamingTV']     == 'Yes').astype(int) +
    (df_feat['StreamingMovies'] == 'Yes').astype(int)
)
df_feat['monthly_to_total_ratio'] = (
    df_feat['MonthlyCharges'] / (df_feat['TotalCharges'] + 1)
)
df_feat['is_month_to_month'] = (df_feat['Contract'] == 'Month-to-month').astype(int)
df_feat['is_fiber']          = (df_feat['InternetService'] == 'Fiber optic').astype(int)
df_feat['autopay'] = df_feat['PaymentMethod'].isin(
    ['Bank transfer (automatic)', 'Credit card (automatic)']
).astype(int)

eng_feats = ['charges_per_month_of_tenure','is_new_customer','is_loyal_customer',
             'has_no_services','n_streaming_services','monthly_to_total_ratio',
             'is_month_to_month','is_fiber','autopay']

print(f"  Original features : {df.shape[1]-1}")
print(f"  Engineered        : {len(eng_feats)}")

for f in eng_feats:
    corr = df_feat[f].corr(df_feat['Churn'])
    print(f"    {f:35s}: corr={corr:+.4f}")


# =============================================================================
# SECTION 5 — PREPROCESSING PIPELINE
# =============================================================================
print("\n[5/8] Building preprocessing pipeline...")

TARGET = 'Churn'
X = df_feat.drop(TARGET, axis=1)
y = df_feat[TARGET]

# Identify column types after engineering
num_cols = ['tenure','MonthlyCharges','TotalCharges',
            'charges_per_month_of_tenure','monthly_to_total_ratio']
bin_cols = ['SeniorCitizen','is_new_customer','is_loyal_customer',
            'has_no_services','n_streaming_services','is_month_to_month',
            'is_fiber','autopay']
cat_cols = ['gender','Partner','Dependents','PhoneService','MultipleLines',
            'InternetService','OnlineSecurity','OnlineBackup','DeviceProtection',
            'TechSupport','StreamingTV','StreamingMovies','Contract',
            'PaperlessBilling','PaymentMethod']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('bin', 'passthrough',    bin_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
], remainder='drop')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc  = preprocessor.transform(X_test)

print(f"  Train: {X_train.shape[0]:,}  Test: {X_test.shape[0]:,}")
print(f"  Features after encoding: {X_train_enc.shape[1]}")
print(f"  Churn rate train: {y_train.mean()*100:.1f}%  "
      f"test: {y_test.mean()*100:.1f}%  (balanced by stratify)")


# =============================================================================
# SECTION 6 — MODEL TRAINING
# =============================================================================
print("\n[6/8] Training 3 models...")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
churn_ratio = (y_train==0).sum() / (y_train==1).sum()

models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=300, max_depth=None, min_samples_leaf=5,
        class_weight='balanced', max_features='sqrt',
        random_state=42, n_jobs=-1),

    'XGBoost': XGBClassifier(
        n_estimators=400, learning_rate=0.05, max_depth=5,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=churn_ratio,
        use_label_encoder=False, eval_metric='logloss',
        verbosity=0, random_state=42),

    'LightGBM': lgb.LGBMClassifier(
        n_estimators=400, learning_rate=0.05, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8,
        class_weight='balanced',
        random_state=42, verbose=-1),
}

results = {}
for name, model in models.items():
    t0 = time.time()
    model.fit(X_train_enc, y_train)
    elapsed = time.time() - t0

    y_pred  = model.predict(X_test_enc)
    y_proba = model.predict_proba(X_test_enc)[:, 1]

    cv_f1 = cross_val_score(model, X_train_enc, y_train,
                             cv=skf, scoring='f1', n_jobs=-1)

    results[name] = {
        'model':      model,
        'y_pred':     y_pred,
        'y_proba':    y_proba,
        'accuracy':   accuracy_score(y_test, y_pred),
        'f1_churn':   f1_score(y_test, y_pred),
        'f1_macro':   f1_score(y_test, y_pred, average='macro'),
        'auc_roc':    roc_auc_score(y_test, y_proba),
        'avg_prec':   average_precision_score(y_test, y_proba),
        'cv_f1_mean': cv_f1.mean(),
        'cv_f1_std':  cv_f1.std(),
        'time_s':     elapsed,
        'cm':         confusion_matrix(y_test, y_pred),
    }
    print(f"\n  {name}:")
    print(f"    F1 (churn class) : {results[name]['f1_churn']:.4f}")
    print(f"    F1 (macro)       : {results[name]['f1_macro']:.4f}")
    print(f"    AUC-ROC          : {results[name]['auc_roc']:.4f}")
    print(f"    CV F1 (5-fold)   : {cv_f1.mean():.4f} ± {cv_f1.std():.4f}")
    print(f"    Training time    : {elapsed:.2f}s")


# =============================================================================
# SECTION 7 — MODEL EVALUATION PLOTS
# =============================================================================
print("\n[7/8] Generating evaluation plots → plots/02_model_evaluation.png ...")

colors_m = {'Random Forest': TEAL, 'XGBoost': BLUE, 'LightGBM': AMBER}

fig, axes = plt.subplots(2, 4, figsize=(22, 11))
fig.suptitle('Model Evaluation — Customer Churn Predictor',
             fontsize=14, fontweight='500', y=1.01)

# ── F1 score comparison ────────────────────────────────────────────────────────
ax = axes[0, 0]
names  = list(results.keys())
f1s    = [results[n]['f1_churn'] for n in names]
cv_f1s = [results[n]['cv_f1_mean'] for n in names]
cv_std = [results[n]['cv_f1_std']  for n in names]
x = np.arange(len(names))
bars = ax.bar(x, f1s, color=[colors_m[n] for n in names],
              edgecolor='white', linewidth=0.5, width=0.6)
ax.errorbar(x, cv_f1s, yerr=cv_std, fmt='o', color='black',
            capsize=4, linewidth=1.5, markersize=5, label='CV F1 ± std')
ax.set_xticks(x); ax.set_xticklabels(names, fontsize=10)
ax.set_ylabel('F1 score (churn class)'); ax.set_ylim(0.5, 1.0)
ax.set_title('F1 score comparison\n(bar=test, dot=CV mean ± std)', fontweight='500')
ax.legend(fontsize=9)
for bar, val in zip(bars, f1s):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{val:.4f}', ha='center', fontsize=10, fontweight='500')

# ── AUC-ROC curves ────────────────────────────────────────────────────────────
ax = axes[0, 1]
for name in names:
    fpr, tpr, _ = roc_curve(y_test, results[name]['y_proba'])
    auc = results[name]['auc_roc']
    ax.plot(fpr, tpr, color=colors_m[name], linewidth=2,
            label=f"{name} (AUC={auc:.3f})")
ax.plot([0,1],[0,1],'k--',linewidth=1,alpha=0.5,label='Random (0.5)')
ax.fill_between(fpr, tpr, alpha=0.05, color=TEAL)
ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate')
ax.set_title('ROC curves — all models', fontweight='500')
ax.legend(fontsize=8, loc='lower right')

# ── Precision-Recall curves ───────────────────────────────────────────────────
ax = axes[0, 2]
for name in names:
    prec, rec, _ = precision_recall_curve(y_test, results[name]['y_proba'])
    ap = results[name]['avg_prec']
    ax.plot(rec, prec, color=colors_m[name], linewidth=2,
            label=f"{name} (AP={ap:.3f})")
ax.axhline(y_test.mean(), color=GRAY, linestyle='--', linewidth=1,
           label=f'Baseline ({y_test.mean():.2f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall curves\n(better for imbalanced data)', fontweight='500')
ax.legend(fontsize=8)

# ── Summary metrics table ─────────────────────────────────────────────────────
ax = axes[0, 3]
ax.axis('off')
metrics_data = [[n,
                 f"{results[n]['f1_churn']:.4f}",
                 f"{results[n]['f1_macro']:.4f}",
                 f"{results[n]['auc_roc']:.4f}",
                 f"{results[n]['accuracy']:.4f}",
                 f"{results[n]['time_s']:.1f}s"]
                for n in names]
table = ax.table(
    cellText=metrics_data,
    colLabels=['Model', 'F1\n(churn)', 'F1\n(macro)', 'AUC\nROC', 'Acc', 'Time'],
    loc='center', cellLoc='center'
)
table.auto_set_font_size(False); table.set_fontsize(9)
table.scale(1, 2.2)
for (r, c), cell in table.get_celld().items():
    if r == 0: cell.set_facecolor('#E1F5EE'); cell.set_text_props(fontweight='bold')
    elif r % 2 == 0: cell.set_facecolor('#FAFAF9')
ax.set_title('Model summary', fontweight='500', pad=15)

# ── Confusion matrices ────────────────────────────────────────────────────────
for i, name in enumerate(names):
    ax = axes[1, i]
    cm = results[name]['cm']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Churn','Churned'],
                yticklabels=['No Churn','Churned'],
                annot_kws={'size': 12}, linewidths=0.5,
                cbar_kws={'shrink': 0.8})
    ax.set_title(f'{name}\nConfusion matrix', fontweight='500')
    ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
    tn,fp,fn,tp = cm.ravel()
    ax.text(0.5,-0.22,
            f'Precision={tp/(tp+fp):.3f}  Recall={tp/(tp+fn):.3f}  F1={results[name]["f1_churn"]:.3f}',
            ha='center', transform=ax.transAxes, fontsize=9, color=GRAY)

# ── Metric radar / bar ────────────────────────────────────────────────────────
ax = axes[1, 3]
metric_names = ['F1 Churn','F1 Macro','AUC-ROC','Accuracy','CV F1']
x = np.arange(len(metric_names))
width = 0.25
for i, name in enumerate(names):
    vals = [results[name]['f1_churn'], results[name]['f1_macro'],
            results[name]['auc_roc'],  results[name]['accuracy'],
            results[name]['cv_f1_mean']]
    ax.bar(x + i*width, vals, width, label=name, color=colors_m[name], alpha=0.85)
ax.set_xticks(x + width); ax.set_xticklabels(metric_names, fontsize=9)
ax.set_ylabel('Score'); ax.set_ylim(0.6, 1.0)
ax.set_title('All metrics compared', fontweight='500')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('plots/02_model_evaluation.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.close()
print("  Saved → plots/02_model_evaluation.png")


# =============================================================================
# SECTION 8 — BUSINESS INSIGHTS
# =============================================================================
print("\n[8/8] Business insights → plots/03_business_insights.png ...")

# Get feature names after encoding
num_feat_names = num_cols
bin_feat_names = bin_cols
cat_feat_names = preprocessor.named_transformers_['cat']\
                              .get_feature_names_out(cat_cols).tolist()
all_feat_names = num_feat_names + bin_feat_names + cat_feat_names

def get_importances(model, feat_names):
    imp = model.feature_importances_
    return pd.Series(imp, index=feat_names[:len(imp)]).sort_values(ascending=False)

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle('Business Insights — What Drives Customer Churn?',
             fontsize=14, fontweight='500', y=1.01)

# ── Feature importances per model ────────────────────────────────────────────
for ax, name, color in zip([axes[0,0], axes[0,1], axes[1,0]],
                           names,
                           [TEAL, BLUE, AMBER]):
    imp = get_importances(results[name]['model'], all_feat_names).head(15)
    ax.barh(imp.index[::-1], imp.values[::-1], color=color, alpha=0.85)
    ax.set_xlabel('Feature importance')
    ax.set_title(f'{name} — top 15 features', fontweight='500')
    ax.tick_params(axis='y', labelsize=9)

# ── Consensus: average importance across all 3 models ─────────────────────────
ax = axes[1, 1]
imps = []
for name in names:
    imp = get_importances(results[name]['model'], all_feat_names)
    imp_norm = imp / imp.sum()
    imps.append(imp_norm)

consensus = pd.concat(imps, axis=1).fillna(0).mean(axis=1).sort_values(ascending=False)
top20 = consensus.head(20)

bar_colors = []
for feat in top20.index:
    feat_lower = feat.lower()
    if any(k in feat_lower for k in ['contract','month_to_month','tenure','loyal']):
        bar_colors.append(TEAL)
    elif any(k in feat_lower for k in ['monthly','charge','fiber','internet']):
        bar_colors.append(CORAL)
    elif any(k in feat_lower for k in ['security','backup','tech','protection','service']):
        bar_colors.append(BLUE)
    else:
        bar_colors.append(AMBER)

ax.barh(top20.index[::-1], top20.values[::-1], color=bar_colors[::-1], alpha=0.85)
ax.set_xlabel('Average normalised importance (3 models)')
ax.set_title('Consensus feature importance\n(averaged across RF + XGBoost + LightGBM)',
             fontweight='500')
ax.tick_params(axis='y', labelsize=9)

# Colour legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(color=TEAL,  label='Contract / Loyalty'),
    Patch(color=CORAL, label='Charges / Internet type'),
    Patch(color=BLUE,  label='Value-added services'),
    Patch(color=AMBER, label='Demographics / Other'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('plots/03_business_insights.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.close()
print("  Saved → plots/03_business_insights.png")


# =============================================================================
# SECTION 9 — MLflow EXPERIMENT TRACKING
# =============================================================================
print("\n[9/9] Logging to MLflow...")

mlflow.set_tracking_uri("sqlite:///mlflow_churn.db")
mlflow.set_experiment("week2_churn_predictor")

best_name = max(names, key=lambda n: results[n]['f1_churn'])

for name in names:
    r = results[name]
    with mlflow.start_run(run_name=name):
        mlflow.log_params({
            "model_type":    name,
            "n_estimators":  models[name].get_params().get('n_estimators'),
            "learning_rate": models[name].get_params().get('learning_rate', 'N/A'),
            "class_balance": "handled"
        })
        mlflow.log_metric("f1_churn",      r['f1_churn'])
        mlflow.log_metric("f1_macro",      r['f1_macro'])
        mlflow.log_metric("auc_roc",       r['auc_roc'])
        mlflow.log_metric("accuracy",      r['accuracy'])
        mlflow.log_metric("cv_f1_mean",    r['cv_f1_mean'])
        mlflow.log_metric("cv_f1_std",     r['cv_f1_std'])
        mlflow.log_metric("avg_precision", r['avg_prec'])
        mlflow.log_metric("train_time_s",  r['time_s'])
        mlflow.set_tag("best_model", str(name == best_name))

        if name == "Random Forest":
            mlflow.sklearn.log_model(r['model'], "model")
        elif name == "XGBoost":
            mlflow.xgboost.log_model(r['model'], "model")
        elif name == "LightGBM":
            mlflow.lightgbm.log_model(r['model'], "model")

        print(f"  {name:20s}  f1={r['f1_churn']:.4f}  auc={r['auc_roc']:.4f}  ✓")

print(f"\nView: mlflow ui → http://127.0.0.1:5000")


# =============================================================================
# FINAL SUMMARY
# =============================================================================
print("\n" + "=" * 65)
print("  RESULTS SUMMARY")
print("=" * 65)
print(f"\n  {'Model':20s}  {'F1 Churn':>10}  {'AUC-ROC':>8}  {'CV F1':>10}")
print("  " + "-" * 53)
for name in names:
    r = results[name]
    marker = " ← BEST" if name == best_name else ""
    print(f"  {name:20s}  {r['f1_churn']:10.4f}  {r['auc_roc']:8.4f}  "
          f"{r['cv_f1_mean']:>5.4f} ± {r['cv_f1_std']:.4f}{marker}")

print(f"\n  Best model: {best_name}")
print(f"\n  CLASSIFICATION REPORT — {best_name}:")
print(classification_report(y_test, results[best_name]['y_pred'],
                             target_names=['No Churn','Churned']))

print("\n  TOP 10 CHURN DRIVERS (consensus across 3 models):")
print("  " + "-" * 53)
for i, (feat, imp) in enumerate(consensus.head(10).items(), 1):
    print(f"  {i:2d}. {feat:35s}: {imp:.4f}")

print("\n  BUSINESS INSIGHTS:")
print("  1. CONTRACT TYPE is the #1 churn predictor — month-to-month")
print("     customers churn at 43% vs 11% (one year) vs 3% (two year).")
print("     → Offer discounts to convert monthly to annual contracts.")
print()
print("  2. TENURE is the second strongest signal — customers who stay")
print("     past 12 months churn at dramatically lower rates.")
print("     → Invest heavily in first-year customer success.")
print()
print("  3. FIBER OPTIC customers churn more than DSL, despite paying more.")
print("     → Investigate fibre service quality and support issues.")
print()
print("  4. MONTHLY CHARGES — higher bills correlate with churn.")
print("     → Identify customers at price-sensitivity thresholds.")
print()
print("  5. LACK OF SECURITY/BACKUP/TECH SUPPORT services increases churn.")
print("     → Bundle these services as a churn prevention package.")
print()
print("  Plots saved to: plots/")
print("=" * 65)

  TELCO CUSTOMER CHURN PREDICTOR
  EDA → Feature Engineering → 3 Models → Business Insights

[1/8] Loading data...
  Source: downloading from GitHub...
  Shape        : 7,043 customers × 21 features
  Churn rate   : 26.5%
  Missing vals : 0

[2/8] Cleaning data...
  TotalCharges NaN (new customers): 11 → filling with MonthlyCharges
  Churn = 1: 1,869  (26.5%)
  Churn = 0: 5,174  (73.5%)
  ⚠  Class imbalance present — will use F1 score and class_weight

[3/8] Running EDA → plots/01_eda_overview.png ...
  Saved → plots/01_eda_overview.png

[4/8] Feature engineering...
  Original features : 19
  Engineered        : 9
    charges_per_month_of_tenure        : corr=+0.0710
    is_new_customer                    : corr=+0.3085
    is_loyal_customer                  : corr=-0.2668
    has_no_services                    : corr=+0.3196
    n_streaming_services               : corr=+0.0712
    monthly_to_total_ratio             : corr=+0.3217
    is_month_to_month                  : corr=+0.4051


2026/07/09 04:37:43 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/09 04:37:43 INFO mlflow.store.db.utils: Updating database tables
2026/07/09 04:37:45 INFO mlflow.tracking.fluent: Experiment with name 'week2_churn_predictor' does not exist. Creating a new experiment.
2026/07/09 04:37:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/09 04:38:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  Random Forest         f1=0.6344  auc=0.8402  ✓


2026/07/09 04:38:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  XGBoost               f1=0.6257  auc=0.8356  ✓
  LightGBM              f1=0.6019  auc=0.8308  ✓

View: mlflow ui → http://127.0.0.1:5000

  RESULTS SUMMARY

  Model                   F1 Churn   AUC-ROC       CV F1
  -----------------------------------------------------
  Random Forest             0.6344    0.8402  0.6346 ± 0.0194 ← BEST
  XGBoost                   0.6257    0.8356  0.6224 ± 0.0173
  LightGBM                  0.6019    0.8308  0.6225 ± 0.0222

  Best model: Random Forest

  CLASSIFICATION REPORT — Random Forest:
              precision    recall  f1-score   support

    No Churn       0.89      0.79      0.84      1035
     Churned       0.56      0.74      0.63       374

    accuracy                           0.78      1409
   macro avg       0.72      0.76      0.74      1409
weighted avg       0.80      0.78      0.78      1409


  TOP 10 CHURN DRIVERS (consensus across 3 models):
  -----------------------------------------------------
   1. Contract_Month-to-mont